In [8]:
# 代码作用：读取LIMS数据汇总为桔水汁水班次数据表 v1
# 2026/03/17
# 数据治理工程师：yushumeng
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType,DateType
from pyspark.sql.functions import current_timestamp,split,lit,col
from pyspark.sql.functions import col, lit, sum, avg, count, when, round,concat
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import to_date
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()

# 读取桔水，汁水榨季累计数据
lims_zjljjs_sql ="""
SELECT 
year,
company,
date,
tzph_cqz,
tzph_lqz,
ctj_cd,
zzjs_zlcd,
zzjs_cd
FROM dwr.dwr_lims_sugar_season_production_company_f_1d 
"""
lims_zjljjs_df = spark.sql(lims_zjljjs_sql)
lims_zjljjs_df = lims_zjljjs_df.withColumnRenamed("season", "zjljjs")



# 读取LIMS班报榨蔗桔水
lims_js_sql ="""
SELECT 
    season,
    gongsidm,
    rq,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        ELSE banbiedm
    END AS banbiedm,
    chanliang,
    gp,
    bx
FROM dwd.dwd_lims_sugar_cane_juice_batch_report_f_1h 
WHERE 
    gongsidm = 'FNR' 
    AND name = '榨蔗桔水'
    AND gp IS NOT NULL
    AND bx IS NOT NULL
    AND chanliang IS NOT NULL;
"""
lims_js_df = spark.sql(lims_js_sql)
lims_js_df = lims_js_df.withColumnRenamed("season", "js_season")


# 读取lims班报糖膏与汁水数据
lims_tgzj_sql = """
SELECT
    season,
    gongsidm,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        ELSE banbiedm
    END AS banbiedm,
    rq,
    MAX(CASE WHEN leibiedm = '澄清汁' THEN ph END) AS clarified_juice_ph,
    MAX(CASE WHEN leibiedm = '滤清汁' THEN ph END) AS filtered_juice_ph,
    MAX(CASE WHEN leibiedm = '粗糖浆' THEN bx END) AS raw_syrup_bx
FROM dwd.dwd_lims_sugar_cane_juice_syrup_batch_report_f_1h
WHERE gongsidm = 'FNR' 
  AND leibiedm IN ('澄清汁', '滤清汁', '粗糖浆')
GROUP BY 
    season,
    gongsidm,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        ELSE banbiedm
    END,
    rq;
"""
lims_tgzj_df = spark.sql(lims_tgzj_sql)
lims_tgzj_df = lims_tgzj_df.withColumnRenamed("season", "tgzj_season")

Spark 启动中...


/opt/tiger/spark/python/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


In [9]:
mes_lims_df_v1 = lims_tgzj_df.join(
    lims_js_df,
    on=[
        lims_tgzj_df["tgzj_season"] == lims_js_df["js_season"],
        lims_tgzj_df["gongsidm"] == lims_js_df["gongsidm"],
        lims_tgzj_df["rq"] == lims_js_df["rq"],
        lims_tgzj_df["banbiedm"] == lims_js_df["banbiedm"]
    ],
    how="inner"
).select(
    lims_tgzj_df["tgzj_season"],         
    lims_tgzj_df["gongsidm"],             
    lims_tgzj_df["rq"],                 
    lims_tgzj_df["banbiedm"],           
    col("clarified_juice_ph"),              
    col("filtered_juice_ph"),
    col("raw_syrup_bx"),
    col("gp"),
    col("bx")
)
mes_lims_df_v1.show()

+-----------+--------+----------+--------+------------------+-----------------+-------------+-------------+-------------+
|tgzj_season|gongsidm|        rq|banbiedm|clarified_juice_ph|filtered_juice_ph| raw_syrup_bx|           gp|           bx|
+-----------+--------+----------+--------+------------------+-----------------+-------------+-------------+-------------+
|      23/24|     FNR|2024-02-07|    乙班|      7.7900000000|     8.5000000000|75.2800000000|40.1100000000|86.1600000000|
|      25/26|     FNR|2026-01-02|    丙班|      7.9050000000|     7.9300000000|68.2800000000|40.4800000000|89.1400000000|
|      22/23|     FNR|2023-02-21|    乙班|      7.8900000000|             null|71.1400000000|39.7000000000|89.1600000000|
|      24/25|     FNR|2024-12-19|    甲班|      7.5400000000|     8.1500000000|72.2200000000|39.8500000000|87.8200000000|
|      24/25|     FNR|2025-03-14|    甲班|      8.5000000000|             null|         null|42.5500000000|88.9200000000|
|      23/24|     FNR|2024-01-09| 

In [10]:
# 5. 写入Hive表
target_table = "app.app_lims_molasses_juice_shift_f_1h"   # 请根据实际数据库名修改

# 添加系统字段
from pyspark.sql.functions import lit, current_timestamp

df_to_write = mes_lims_df_v1 \
    .withColumn("source_flg", lit("LIMS+MES")) \
    .withColumn("dw_update_time", current_timestamp())

df_to_write.write \
    .mode("overwrite") \
    .format("hive") \
    .option("encoding", "UTF-8") \
    .saveAsTable(target_table)

# 6. 添加表级注释
spark.sql(f"""
    ALTER TABLE {target_table} 
    SET TBLPROPERTIES (
        'comment' = '桔水汁水班次数据表',
        'create_time' = '{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
    )
""")

# 7. 定义字段注释（根据实际字段调整）
field_comments = {
    "tgzj_season": "榨季",
    "gongsidm": "公司代码",
    "rq": "日期",
    "banbiedm": "班别代码",
    "clarified_juice_ph": "清汁PH值",
    "filtered_juice_ph": "滤汁PH值",
    "raw_syrup_bx": "粗糖浆锤度",
    "gp": "废蜜重力纯度",
    "bx": "桔水锤度",
    "source_flg": "数据来源标识（LIMS+MES）",
    "dw_update_time": "数仓更新时间"
}

# 8. 工具函数：转换Spark类型为Hive类型字符串
def get_hive_type_str(field):
    data_type = field.dataType
    type_name = type(data_type).__name__
    if type_name == "StringType":
        return "STRING"
    elif type_name == "DateType":
        return "DATE"
    elif type_name == "TimestampType":
        return "TIMESTAMP"
    elif type_name == "DoubleType":
        return "DOUBLE"
    elif type_name == "DecimalType":
        return f"DECIMAL({data_type.precision},{data_type.scale})"
    else:
        return "STRING"

# 9. 批量添加字段注释（从DataFrame获取schema）
field_type_map = {f.name: get_hive_type_str(f) for f in df_to_write.schema.fields}
# 由于已经添加了系统字段，field_type_map已包含它们，无需额外处理

for field, comment in field_comments.items():
    spark.sql(f"""
        ALTER TABLE {target_table} 
        CHANGE COLUMN {field} {field} {field_type_map.get(field, "STRING")} COMMENT '{comment}'
    """)

# 10. 输出执行结果
print(f"✅ 数据已成功写入Hive表：{target_table}")
print(f"📝 表注释：桔水汁水班次数据表（包含班次级别的桔水、汁水关键指标）")
print(f"🕒 写入时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 可选：停止Spark会话
spark.stop()

✅ 数据已成功写入Hive表：app.app_lims_molasses_juice_shift_f_1h
📝 表注释：桔水汁水班次数据表（包含班次级别的桔水、汁水关键指标）
🕒 写入时间：2026-03-20 16:29:05
